# Workflow Capacity Explorer

Сравнение **монолит vs sharding**: wait / work / total по срезам.

**Run All** — всё ниже работает без ручной настройки. Переменные с комментарием «можно менять» — опциональные override.

- Jobs кэшируются в `data/cache/` — повторные прогоны быстрые
- Первый запуск без кэша скачает данные с GitHub (`gh auth login`)
- Конфиг: `config/capacity.example.yml`
- HTML-страница сравнения: [generate_comparison_page.ipynb](./generate_comparison_page.ipynb)

In [1]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from workflow_capacity.cache import ensure_dataset, list_datasets, load_dataset, resolve_dataset
from workflow_capacity.config import PoolConfig
from workflow_capacity.compare import (
    evaluate_config,
    results_to_dataframe,
    sankey_wait_work_flow,
    sankey_compare_configs,
)
from workflow_capacity.metrics import (
    D_GROUPS,
    agg_get,
    comparison_table,
    pct_label,
    scenario_metrics,
)
from workflow_capacity.pr_check import build_pr_check_runs
from workflow_capacity.pr_classify import ClassifyRules
from workflow_capacity.simulate import run_pair

CONFIG_PATH = ROOT / 'config' / 'capacity.example.yml'
CACHE_DIR = ROOT / 'data' / 'cache'
PEAK_HOURS = list(range(9, 16))

# можно менять:
PERCENTILES = [90]           # одно число или список, напр. [50, 90, 95]
PRIMARY_PERCENTILE = 90      # для сводок, Sankey и колонок по умолчанию
PCT_COL = pct_label(PRIMARY_PERCENTILE)

## 1. Исторические данные

Авто: последний файл в `data/cache/`, иначе скачивание с GitHub.

In [ ]:
# можно менять:
DAYS = 7
REPO = 'ydb-platform/ydb'
REFRESH = True          # True = принудительно перекачать с GitHub
SELECTED_CACHE = None    # None = авто; или Path('data/cache/jobs_....json')

cached = list_datasets(CACHE_DIR)
print('Cached windows:')
if cached:
    for ds in cached:
        print(f'  - {ds.path.name} ({len(ds.jobs)} jobs, {ds.since[:10]} .. {ds.until[:10]})')
else:
    print('  (пусто — будет скачивание с GitHub)')

dataset = resolve_dataset(
    days=DAYS,
    repo=REPO,
    cache_dir=CACHE_DIR,
    refresh=REFRESH,
    selected=SELECTED_CACHE,
)

print(f'\nActive: {dataset.path.name}')
print(f'Jobs: {len(dataset.jobs)} | {dataset.since[:10]} .. {dataset.until[:10]}')

Cached windows:
  (пусто — будет скачивание с GitHub)
cache: WARNING — only incomplete jobs_ydb-platform_ydb_2026-06-07_2026-06-14.json.partial.json (finish collect or set REFRESH=True for full window)

Active: jobs_ydb-platform_ydb_2026-06-07_2026-06-14.json.partial.json
Jobs: 2101 | 2026-06-07 .. 2026-06-14


### Сравнение двух окон (опционально)

Авто: если в кэше ≥2 файлов — сравнивает два последних. Иначе пропускает.

In [3]:
# можно менять: None = авто, True = принудительно, False = пропустить
COMPARE_WINDOWS = None

windows = list_datasets(CACHE_DIR)
do_compare = len(windows) >= 2 if COMPARE_WINDOWS is None else COMPARE_WINDOWS

if do_compare and len(windows) >= 2:
    ds_a, ds_b = windows[-2], windows[-1]
    print(f'Compare: {ds_a.path.name} vs {ds_b.path.name}')
    cfg_cmp = PoolConfig.load(CONFIG_PATH, name='current')
    pr_a = build_pr_check_runs(ds_a.jobs, classify=False)
    pr_b = build_pr_check_runs(ds_b.jobs, classify=False)
    ra = evaluate_config(
        ds_a.jobs, pr_a, cfg_cmp, peak_hours=PEAK_HOURS,
        percentiles=PERCENTILES, primary_percentile=PRIMARY_PERCENTILE,
    )
    rb = evaluate_config(
        ds_b.jobs, pr_b, cfg_cmp, peak_hours=PEAK_HOURS,
        percentiles=PERCENTILES, primary_percentile=PRIMARY_PERCENTILE,
    )
    def _totals(item):
        b = item.base_agg[(None, 'all')]
        p = item.par_agg[(None, 'all')]
        mt = agg_get(b, 'total', PRIMARY_PERCENTILE)
        st = agg_get(p, 'total', PRIMARY_PERCENTILE)
        return mt, st, (st or 0) - (mt or 0)
    ma, sa, da = _totals(ra)
    mb, sb, db = _totals(rb)
    cmp_df = pd.DataFrame([
        {
            'window': ds_a.path.stem,
            'jobs': len(ds_a.jobs),
            f'mono_total_{PCT_COL}': ma,
            f'shard_total_{PCT_COL}': sa,
            'delta_total': da,
        },
        {
            'window': ds_b.path.stem,
            'jobs': len(ds_b.jobs),
            f'mono_total_{PCT_COL}': mb,
            f'shard_total_{PCT_COL}': sb,
            'delta_total': db,
        },
    ]).round(1)
    display(cmp_df)
elif do_compare:
    print('Сравнение: нужно ≥2 файлов в data/cache/ (сейчас', len(windows), ')')
else:
    print('Сравнение окон: пропущено (в кэше', len(windows), 'файл(ов))')

Сравнение окон: пропущено (в кэше 0 файл(ов))


## 2. Конфигурации capacity

Базовый конфиг + сценарии масштабирования квот.
Редактируйте `config/capacity.example.yml` для static runners и footprints.

In [4]:
base = PoolConfig.load(CONFIG_PATH, name='current')

CONFIGS = [
    base,
    base.with_quota_scale(vcpu=1.1, name='vcpu+10%'),
    base.with_quota_scale(vcpu=1.25, instances=1.25, ram_gb=1.25, nrd_ssd_gb=1.25, name='all+25%'),
    base.with_quota_scale(vcpu=2.0, instances=2.0, ram_gb=2.0, nrd_ssd_gb=2.0, name='all x2'),
]

pd.DataFrame([
    {
        'config': c.name,
        'instances': int(c.quotas['instances']),
        'vcpu': int(c.quotas['vcpu']),
        'ram_gb': int(c.quotas['ram_gb']),
        'vm_budget': round(c.max_instances_budget(), 1),
    }
    for c in CONFIGS
])

,config,instances,vcpu,ram_gb,vm_budget
0,current,110,5400,23000,73.8
1,vcpu+10%,110,5940,23000,73.8
2,all+25%,137,6750,28750,98.1
3,all x2,220,10800,46000,172.8


## 3. Прогон симуляций

PR-check = **max(relwithdebinfo, release-asan)**; в шардинге оба job через prepare + shards.

`CLASSIFY=True` — sharded/single по **закэшированным** путям PR (`pr_files` в jobs_*.json, правила в `config/capacity.example.yml` → `pr_classify`).

In [5]:
# можно менять:
CLASSIFY = True   # False = все PR sharded (верхняя оценка)
ROLLOUT = 'all eligible'

from workflow_capacity.metrics import ROLL_OUTS
rollout = next(r for r in ROLL_OUTS if r[1] == ROLLOUT)
_, rollout_label, shard_eligible = rollout

classify_rules = ClassifyRules.load(CONFIG_PATH)
cached_prs = len(dataset.pr_files)
print(f'Building PR-check runs (classify={CLASSIFY}, pr_files in cache: {cached_prs}) ...', flush=True)
pr_runs = build_pr_check_runs(
    dataset.jobs,
    classify=CLASSIFY,
    pr_files=dataset.pr_files,
    classify_rules=classify_rules,
)
print(f'PR-check runs: {len(pr_runs)} (sharded={sum(1 for r in pr_runs if r.mode == "sharded")})', flush=True)

results = []
for i, cfg in enumerate(CONFIGS, 1):
    print(f'[{i}/{len(CONFIGS)}] simulate: {cfg.name} ...', flush=True)
    results.append(
        evaluate_config(
            dataset.jobs,
            pr_runs,
            cfg,
            rollout_label=rollout_label,
            shard_eligible=shard_eligible,
            peak_hours=PEAK_HOURS,
            percentiles=PERCENTILES,
            primary_percentile=PRIMARY_PERCENTILE,
        )
    )
print('Simulations done.', flush=True)

df = results_to_dataframe(results)
summary = df.groupby('config').agg(
    mono_wait=(f'mono_wait_{PCT_COL}', 'median'),
    mono_work=(f'mono_work_{PCT_COL}', 'median'),
    mono_total=(f'mono_total_{PCT_COL}', 'median'),
    shard_wait=(f'shard_wait_{PCT_COL}', 'median'),
    shard_work=(f'shard_work_{PCT_COL}', 'median'),
    shard_total=(f'shard_total_{PCT_COL}', 'median'),
    delta_total=('delta_total', 'median'),
).round(1)
display(summary)

Building PR-check runs (classify=True, pr_files in cache: 0) ...
PR-check runs: 379 (sharded=0)
[1/4] simulate: current ...
[2/4] simulate: vcpu+10% ...
[3/4] simulate: all+25% ...
[4/4] simulate: all x2 ...
Simulations done.


,mono_wait,mono_work,mono_total,shard_wait,shard_work,shard_total,delta_total
config,,,,,,,
all x2,0.0,199.0,199.0,0.0,199.0,199.0,0.0
all+25%,0.0,199.0,199.0,0.0,199.0,199.0,0.0
current,0.0,199.0,199.0,0.0,199.0,199.0,0.0
vcpu+10%,0.0,199.0,199.0,0.0,199.0,199.0,0.0


## 4. Таблица по срезам (час × D)

In [6]:
# можно менять:
CONFIG_NAME = 'current'
D_FILTER = 'все D'

view = df[(df['config'] == CONFIG_NAME) & (df['d_group'] == D_FILTER)].copy()
cols = [
    'hour_utc', 'd_group', 'n',
    f'mono_wait_{PCT_COL}', f'shard_wait_{PCT_COL}', 'delta_wait',
    f'mono_work_{PCT_COL}', f'shard_work_{PCT_COL}', 'delta_work',
    f'mono_total_{PCT_COL}', f'shard_total_{PCT_COL}', 'delta_total', 'delta_total_pct',
]
display(view[cols].round(1))

,hour_utc,d_group,n,mono_wait_p90,shard_wait_p90,delta_wait,mono_work_p90,shard_work_p90,delta_work,mono_total_p90,shard_total_p90,delta_total,delta_total_pct
4,09:00,все D,7,0.0,0.0,0,197.8,197.8,0.0,197.8,197.8,0.0,0.0
9,10:00,все D,14,0.0,0.0,0,202.1,202.1,0.0,202.1,202.1,0.0,0.0
14,11:00,все D,11,0.0,0.0,0,199.7,199.7,0.0,199.7,199.7,0.0,0.0
19,12:00,все D,17,0.0,0.0,0,4.7,4.7,0.0,4.7,4.7,0.0,0.0
24,13:00,все D,14,0.0,0.0,0,209.2,209.2,0.0,209.2,209.2,0.0,0.0
29,14:00,все D,14,0.0,0.0,0,205.7,205.7,0.0,205.7,205.7,0.0,0.0
34,15:00,все D,4,0.0,0.0,0,240.8,240.8,0.0,240.8,240.8,0.0,0.0


## 5. Sankey: монолит → шардинг (wait / work)

In [7]:
item = next(r for r in results if r.config.name == CONFIG_NAME)
row = item.base_agg.get((None, 'all'), {})
prow = item.par_agg.get((None, 'all'), {})

fig = go.Figure(data=[go.Sankey(
    node=dict(
        label=['Монолит wait', 'Монолит work', 'Шардинг wait', 'Шардинг work'],
        color=['#e45756', '#4c78a8', '#f58518', '#72b7b2'],
    ),
    link=dict(
        source=[0, 1],
        target=[2, 3],
        value=[
            max(agg_get(prow, 'wait', PRIMARY_PERCENTILE) or 0, 0.1),
            max(agg_get(prow, 'work', PRIMARY_PERCENTILE) or 0, 0.1),
        ],
    ),
)])
fig.update_layout(
    title=f'{CONFIG_NAME}: {PCT_COL} wait/work (все D, пик 09-15) — '
            f"mono total {agg_get(row, 'total', PRIMARY_PERCENTILE) or 0:.0f} → "
            f"shard {agg_get(prow, 'total', PRIMARY_PERCENTILE) or 0:.0f} min",
    height=420,
)
fig.show()

## 6. Sankey: сравнение двух конфигов (sharding total)

In [8]:
# можно менять:
LEFT = 'current'
RIGHT = 'all+25%'
HOUR = 10
D_KEY = 'all'

left = next(r for r in results if r.config.name == LEFT)
right = next(r for r in results if r.config.name == RIGHT)
sk = sankey_compare_configs(left, right, hour=HOUR, d_key=D_KEY)

fig = go.Figure(data=[go.Sankey(node=dict(label=sk['node']['label']), link=sk['link'])])
fig.update_layout(
    title=f'Sharding {PCT_COL} @ {HOUR:02d}:00, D={D_KEY}: {LEFT} total {sk["meta"]["left_total"]:.0f} min',
    height=380,
)
fig.show()
print(f'{RIGHT}: wait {sk["meta"]["right_wait"]:.1f}, work {sk["meta"]["right_work"]:.1f}, total {sk["meta"]["right_total"]:.1f} min')

all+25%: wait 0.0, work 202.1, total 202.1 min


## 7. Интерактивные слайдеры квот

In [9]:
import ipywidgets as w

vcpu_mult = w.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.05, description='vCPU x')
inst_mult = w.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.05, description='VM max x')
ram_mult = w.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.05, description='RAM x')
ssd_mult = w.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.05, description='SSD x')
out = w.Output()


def rerun(_=None):
    cfg = base.with_quota_scale(
        vcpu=vcpu_mult.value,
        instances=inst_mult.value,
        ram_gb=ram_mult.value,
        nrd_ssd_gb=ssd_mult.value,
        name='interactive',
    )
    item = evaluate_config(
        dataset.jobs, pr_runs, cfg,
        rollout_label=rollout_label,
        shard_eligible=shard_eligible,
        peak_hours=PEAK_HOURS,
        percentiles=PERCENTILES,
        primary_percentile=PRIMARY_PERCENTILE,
    )
    rows = []
    for d_key, d_label in D_GROUPS:
        b = item.base_agg.get((None, d_key), {})
        p = item.par_agg.get((None, d_key), {})
        mt = agg_get(b, 'total', PRIMARY_PERCENTILE)
        st = agg_get(p, 'total', PRIMARY_PERCENTILE)
        rows.append({
            'd_group': d_label,
            'mono_wait': agg_get(b, 'wait', PRIMARY_PERCENTILE),
            'shard_wait': agg_get(p, 'wait', PRIMARY_PERCENTILE),
            'mono_total': mt,
            'shard_total': st,
            'delta_total': (st or 0) - (mt or 0),
        })
    with out:
        out.clear_output(wait=True)
        display(pd.DataFrame(rows).round(1))
        print(
            f"vm_budget={cfg.max_instances_budget():.1f}, "
            f"vcpu={int(cfg.quotas['vcpu'])}, "
            f"ram={int(cfg.quotas['ram_gb'])}, ssd={int(cfg.quotas['nrd_ssd_gb'])}"
        )

for slider in (vcpu_mult, inst_mult, ram_mult, ssd_mult):
    slider.observe(rerun, names='value')

display(w.VBox([vcpu_mult, inst_mult, ram_mult, ssd_mult, out]))
rerun()